# MCP: Model Context Protocol

*AI Lab course — Class 8: Servers, Clients, and Tools That Travel*

In Lesson 4 we wrote tools in Python and handed them to the model by hand. In Lesson 5 LangChain
packaged that same loop. In both cases the tools lived **inside the application**: if another program
wanted to use them, you copied the code.

**MCP (Model Context Protocol)** solves exactly that. It standardizes how a program publishes tools,
resources, and prompts, so **any** MCP-compatible host can discover and use them — without copying a
single line.

This class builds a real MCP server and a real MCP client, and then connects them to the tool-calling
loop from Lesson 4.

## Learning Objectives

By the end of this notebook, you should be able to:

1. Explain what MCP is and which problem it solves.
2. Describe the client/server architecture and the role of the transport.
3. Build an MCP **server** that exposes tools, a resource, and a prompt.
4. Inspect the **tool schemas** a server publishes, and compare them with the schemas we wrote by hand in Lesson 4.
5. Build an MCP **client** that lists tools and calls them.
6. Convert MCP tools into the OpenAI tool format and plug them into a tool-calling loop.
7. Understand the security implications of exposing tools to an external host.

This notebook uses the official **MCP Python SDK**, the LiteLLM proxy for the model calls, and no
frameworks beyond that. The tool-calling loop is the one from Lesson 4.

## 1. Why do we need MCP?

By Lesson 7 we could build an agent that retrieves, grades its own context, retries, and answers.
Its tools, however, were **hardcoded inside the notebook**:

```text
Lessons 4, 5 and 7 — tools inside the application

  your program
     |
     v
  Python function  ->  tool schema  ->  model requests it  ->  your code runs it
```

That works, but it means:

- The tools only exist in **that** program. VS Code, Claude Desktop, or a teammate's agent cannot use them.
- Every host re-implements discovery and calling in its own way.
- Sharing a tool means sharing code, not connecting to a service.

MCP turns the tools into a **service with a standard interface**:

```text
MCP — tools as a service

  any host (VS Code, Claude Desktop, your own client, ...)
     |
     v
  MCP protocol (discover, then call)
     |
     v
  your MCP server  ->  your Python functions
```

**Important:** MCP is a protocol, not a library and not a model feature. It defines the messages that a
client and a server exchange. The model still does not execute anything — it *requests*, and the host
decides what to run. That separation is the same one we established in Lesson 4; MCP only standardizes
the plumbing around it.

## 2. The architecture: hosts, clients, servers, transports

MCP has three roles, plus a transport:

| Piece | What it is | Examples |
| --- | --- | --- |
| **Host** | The application the user actually uses. It decides when to ask the model, and what to run. | VS Code, Claude Desktop, your own app |
| **Client** | The component **inside** the host that speaks MCP to one server. | `mcp.Client` (the Python class we will use) |
| **Server** | The program that publishes tools, resources, and prompts. | `training_server.py`, which we are about to write |
| **Transport** | How bytes travel between client and server. | `stdio`, `streamable-http` |

A host can hold several clients, one per server:

```text
HOST
 +-- client A  ---stdio--->  server: training operations
 +-- client B  ---stdio--->  server: files
 +-- client C  --http----->  server: company wiki
```

### What a server can expose

MCP defines three main kinds of things a server can publish. The distinction matters:

| Kind | Who decides to use it | Typical use |
| --- | --- | --- |
| **Tools** | The **model** requests them | Calculate, query, create a record |
| **Resources** | The **application** reads them | Documents, catalogue data, config |
| **Prompts** | The **user** picks them | A reusable prompt template with arguments |

Tools are what we already know from Lesson 4. Resources and prompts are the new surface: data a server
publishes as readable content, and templates a host can show in a menu. We will build one of each.

### The transport we will use: `stdio`

```text
stdio transport

  host launches training_server.py as a subprocess
        |                                              ^
        |  JSON-RPC on the child's stdin               |  JSON-RPC on the child's stdout
        v                                              |
  your server process
```

`stdio` is the default and the most common transport for **local** servers: the host starts your file as
a child process and speaks to it over stdin/stdout. There is no port and no URL. Streamable HTTP is the
transport for servers you **deploy**; it is out of scope today.

## 3. Setup

We need the MCP SDK. The `cli` extra adds the `mcp` command-line tool (`mcp dev` runs your server under
the MCP Inspector) on top of the SDK.

Run the next cell once. If it is already installed, it will just confirm it.

In [53]:
# Run once if needed, then restart the kernel.

%pip install -q -U "mcp[cli]"

# The model calls still go through the LiteLLM proxy — no OpenAI key required.
%pip install -q python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


The next cell imports everything we need and prints the versions, so the notebook tells us where it is
running instead of us assuming it.

Note the two different clients in this notebook, which are easy to confuse:

| Object | Talks to | Comes from |
| --- | --- | --- |
| `llm_client` | the model, through the LiteLLM proxy | `utils/open_ai.py` (Lesson 2 onwards) |
| `mcp.Client` | an MCP **server** | the MCP SDK |

They are unrelated, and both run in this notebook.

In [54]:
import json
import os
import subprocess
import sys
from importlib.metadata import version

from dotenv import load_dotenv

load_dotenv(override=False)

from utils.open_ai import OpenAI

llm_client = OpenAI()
MODEL = os.environ.get('MODEL', 'gpt-4.1-mini')

from mcp import Client, StdioServerParameters
from mcp.server import MCPServer

print('mcp SDK    :', version('mcp'))
print('Python     :', sys.version.split()[0])
print('Model      :', MODEL)
print()
print('llm_client is ready (routing to the LiteLLM proxy).')

mcp SDK    : 2.3.0
Python     : 3.12.8
Model      : gpt-4.1-mini

llm_client is ready (routing to the LiteLLM proxy).


### A note on SDK versions (read this once)

The MCP Python SDK shipped a **v2** that renamed its high-level server class. Before v2 the class was
called `FastMCP` and lived in `mcp.server.fastmcp`; since v2 it is called `MCPServer` and lives in
`mcp.server`:

```python
# v1 (old tutorials, blog posts, older repositories)
from mcp.server.fastmcp import FastMCP
mcp = FastMCP('Demo')

# v2 (what `pip install mcp` gives you today)
from mcp.server import MCPServer
mcp = MCPServer('Demo')
```

The decorators (`@mcp.tool()`, `@mcp.resource()`, `@mcp.prompt()`) did not change, and the tool schema is
still derived from your type hints and docstrings. If a tutorial you find uses `FastMCP`, that rename is
the only translation you need.

The cell below proves which generation is installed, instead of assuming it.

In [55]:
# Which SDK generation are we on? (v1 = FastMCP, v2 = MCPServer)
try:
    from mcp.server import MCPServer

    print('SDK generation: v2 — the high-level class is MCPServer.')
except ImportError:
    print('SDK generation: v1 — the high-level class is FastMCP.')
    print('This notebook works the same; the class name in Step 1 is the only change.')

SDK generation: v2 — the high-level class is MCPServer.


# Hands-on

We will build a small **training-operations server** for the AI Lab, using the same fictional domain as
the rest of the course. It will expose:

- **tools**: `add_numbers`, `multiply_numbers`, `divide_numbers`, `get_room_capacity`
- **resource**: `training://catalog` — the course catalogue, as readable data
- **prompt**: `training_plan` — a reusable template with an argument

All the code we write from now on goes into files **next to this notebook**, in the same folder.

## Step 1: The MCP server

### File layout

```text
8-mcp-servers-and-clients/
  8_mcp_servers_and_clients.ipynb   <- you are here
  training_server.py                <- the MCP server we are about to write
  mcp_client.py                     <- the MCP client (Step 2)
  utils/open_ai.py                  <- LiteLLM wrapper, same as previous lessons
```

### The shape of a server

```text
training_server.py

  mcp = MCPServer('Training Ops')            <- the server object, with its name

  @mcp.tool()   def add_numbers(...)         <- the MODEL can request these
  @mcp.tool()   def get_room_capacity(...)

  @mcp.resource('training://catalog')        <- the APPLICATION can read this
  @mcp.prompt() def training_plan(...)       <- the USER can pick this

  if __name__ == '__main__':
      mcp.run()                              <- blocks, speaking JSON-RPC on stdio
```

Notice what we do **not** write: no JSON schema by hand, no request parsing, no validation code, no
dispatch table. The type hints and the docstring *are* the schema — the same input the `@tool` decorator
consumed in Lesson 5 — except now they are served over a protocol instead of living inside our program.

The next cell **writes the file** (the same `%%writefile` pattern we used in Lesson 1).

In [56]:
%%writefile training_server.py
'''MCP server for AI Lab training operations.

Exposes tools, one resource, and one prompt over the stdio transport.

Run it directly for a syntax check:  python training_server.py
(it will block, waiting for a host to speak first — that is correct)
'''

from mcp.server import MCPServer

mcp = MCPServer(
    'Training Ops',
    instructions='Training operations for the AI Lab. Use the math tools for any calculation.',
)


# ---------------------------------------------------------------------------
# Tools — the model can request these
# ---------------------------------------------------------------------------

@mcp.tool()
def add_numbers(a: float, b: float) -> float:
    '''Add two numbers and return the result.'''
    return a + b


@mcp.tool()
def multiply_numbers(a: float, b: float) -> float:
    '''Multiply two numbers and return the result.'''
    return a * b


@mcp.tool()
def divide_numbers(a: float, b: float) -> dict:
    '''Divide a by b and return the result.'''
    if b == 0:
        # Same idea as Lesson 4: a tool that fails returns a structured error
        # the model can read and recover from, instead of crashing the call.
        return {'error': 'Cannot divide by zero. Ask the user for a non-zero divisor.'}
    return {'result': a / b}


ROOMS = {
    'lab_a': {'capacity': 12, 'has_projector': True},
    'lab_b': {'capacity': 24, 'has_projector': True},
    'lab_c': {'capacity': 6, 'has_projector': False},
}


@mcp.tool()
def get_room_capacity(room_id: str) -> dict:
    '''Gets info.'''  ### FIXME: would a model pick this tool with a description like this?
    room = ROOMS.get(room_id)

    if room is None:
        return {'found': False, 'room_id': room_id, 'error': 'Room not found.'}

    return {
        'found': True,
        'room_id': room_id,
        'capacity': room['capacity'],
        'has_projector': room['has_projector'],
    }


# ---------------------------------------------------------------------------
# Resource — the application can read this (no model involved)
# ---------------------------------------------------------------------------

@mcp.resource('training://catalog')
def training_catalog() -> str:
    '''The AI Lab course catalogue.'''
    return (
        '1. Web and FastAPI\n'
        '2. LLM Inference Parameters\n'
        '3. RAG from Scratch\n'
        '4. Function Calling and Tool Usage\n'
        '5. LangChain Tool Calling\n'
        '6. LangGraph Calculator\n'
        '7. LangGraph RAG Agent\n'
    )


# ---------------------------------------------------------------------------
# Prompt — the user can pick this
# ---------------------------------------------------------------------------

@mcp.prompt()
def training_plan(topic: str) -> str:
    '''Draft a short training plan for one topic.'''
    return (
        f'Draft a 90-minute training plan for the topic: {topic}. '
        'Include learning objectives, one hands-on exercise, and one discussion question.'
    )


if __name__ == '__main__':
    # No arguments means stdio: the host launches this file as a subprocess.
    mcp.run()

Overwriting training_server.py


Before running anything, let's check the file is even valid. A server that does not import is a server
that publishes nothing — and the `__main__` guard means importing it will not start the loop.

In [57]:
import ast
import importlib.util

source = open('training_server.py').read()
ast.parse(source)
print('1. The file parses as valid Python.')

spec = importlib.util.spec_from_file_location('training_server', 'training_server.py')
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)

print(f'2. The server object imports: type = {type(module.mcp).__name__}')
print('   Importing is not serving: mcp.run() starts the stdio loop,')
print('   and it sits under the __main__ guard so an import never blocks.')

1. The file parses as valid Python.
2. The server object imports: type = MCPServer
   Importing is not serving: mcp.run() starts the stdio loop,
   and it sits under the __main__ guard so an import never blocks.


> **Checkpoint:** `training_server.py` should now parse, import, and publish four tools, one resource,
> and one prompt. It is not running yet — nobody has launched it.

### Try it in the Inspector (optional, one terminal command)

The SDK ships a command-line tool that does exactly what a real host does: launches your file as a
subprocess and speaks MCP to it over stdio.

```bash
mcp dev training_server.py
```

That opens the **MCP Inspector** in the browser, where you can list the tools, call them by hand, and read
the resource. It needs `npx` on your PATH (the Inspector is a Node.js app), so if it does not start, skip
it — the client we are about to write shows the same information and needs nothing extra.

## Step 2: The client

A client is one object with one lifecycle:

```python
async with Client(transport) as client:   # connect and negotiate
    await client.list_tools()             # use
# leaving the block disconnects
```

For a local server, the transport is described by the command that launches it:

```python
server = StdioServerParameters(
    command=sys.executable,          # the same Python that runs this notebook
    args=['training_server.py'],
)
```

> **Careful:** a stdio child process does **not** inherit your environment. It gets a minimal allow-list
> (`PATH`, `HOME`, `SHELL`, ...) so nothing sensitive leaks into a process you may not have written. Our
> server is self-contained so it needs no variables — but this is why a server that needs an API key must
> receive it explicitly, through `env=`.

The client file below does four things: tools (what the model would see), resources (what the application
can read), prompts (what the user can pick), and tool calls.

In [58]:
%%writefile mcp_client.py
'''Minimal MCP client: connect to training_server.py, list, and call.'''

import asyncio
import sys

from mcp import Client, StdioServerParameters
from mcp.types import TextContent, TextResourceContents

SERVER = StdioServerParameters(
    command=sys.executable,
    args=['training_server.py'],
)


async def main() -> None:
    # Entering the block spawns the subprocess and negotiates; leaving it shuts it down.
    async with Client(SERVER) as client:
        name = client.server_info.name if client.server_info else 'unknown'
        print('Connected to:', name)
        print('Protocol version:', client.protocol_version)
        print('Instructions:', client.instructions)
        print()

        # ---- Tools: what the model would see -------------------------------
        listed = await client.list_tools()
        print(f'Tools available ({len(listed.tools)}):')
        for tool in listed.tools:
            print(f'  - {tool.name}: {tool.description}')
        print()

        # ---- Resources: what the application can read ----------------------
        resources = await client.list_resources()
        print('Resources:', [resource.uri for resource in resources.resources])

        read = await client.read_resource('training://catalog')
        print('Catalog read from the server:')
        for contents in read.contents:
            if isinstance(contents, TextResourceContents):
                print(contents.text)
        print()

        # ---- Prompts: what the user can pick -------------------------------
        prompts = await client.list_prompts()
        print('Prompts:', [prompt.name for prompt in prompts.prompts])

        rendered = await client.get_prompt('training_plan', {'topic': 'MCP'})
        for message in rendered.messages:
            if isinstance(message.content, TextContent):
                print(f'  [{message.role}] {message.content.text}')
        print()

        # ---- Calling a tool -------------------------------------------------
        ok = await client.call_tool('get_room_capacity', {'room_id': 'lab_b'})
        print('call_tool(get_room_capacity, lab_b)')
        print('  is_error          :', ok.is_error)
        print('  structured_content:', ok.structured_content)
        print()

        missing = await client.call_tool('get_room_capacity', {'room_id': 'lab_z'})
        print('call_tool(get_room_capacity, lab_z)')
        print('  is_error          :', missing.is_error)
        print('  structured_content:', missing.structured_content)
        print()

        unknown = await client.call_tool('not_a_tool', {})
        print('call_tool(not_a_tool)')
        print('  is_error:', unknown.is_error)
        print('  content :', unknown.content[0].text)


if __name__ == '__main__':
    asyncio.run(main())

Overwriting mcp_client.py


### Run the client

We run it as a **separate process**, which is exactly how a host runs it. Note the three topics it
exercises: a tool that succeeds, a tool whose argument does not exist, and a tool the server does not even
have.

In [59]:
completed = subprocess.run(
    [sys.executable, 'mcp_client.py'],
    capture_output=True,
    text=True,
    timeout=180,
)

print(completed.stdout)

if completed.returncode != 0:
    print('--- stderr ---')
    print(completed.stderr[-2000:])

Connected to: Training Ops
Protocol version: 2026-07-28
Instructions: Training operations for the AI Lab. Use the math tools for any calculation.

Tools available (4):
  - add_numbers: Add two numbers and return the result.
  - multiply_numbers: Multiply two numbers and return the result.
  - divide_numbers: Divide a by b and return the result.
  - get_room_capacity: Gets info.

Resources: ['training://catalog']
Catalog read from the server:
1. Web and FastAPI
2. LLM Inference Parameters
3. RAG from Scratch
4. Function Calling and Tool Usage
5. LangChain Tool Calling
6. LangGraph Calculator
7. LangGraph RAG Agent


Prompts: ['training_plan']
  [user] Draft a 90-minute training plan for the topic: MCP. Include learning objectives, one hands-on exercise, and one discussion question.

call_tool(get_room_capacity, lab_b)
  is_error          : False
  structured_content: None

call_tool(get_room_capacity, lab_z)
  is_error          : False
  structured_content: None

call_tool(not_a_tool)
 

Notice the last two blocks. Neither a missing room nor a tool that does not exist raised an exception in
the client: both came back as **ordinary results** with `is_error` set. That is the same contract you met
in Lesson 4 — a tool failure is part of the conversation, not a crash — and it survives the protocol.

> **Checkpoint:** you have now done over a real protocol what Lesson 4 did inside a single process:
> discovered tools, read their schemas, called them, and read the results. The difference is that the
> caller never imported the server's code.

## Step 3: See the schemas the server published

In Lesson 4 we wrote schemas by hand:

```python
multiply_tool = {
    'type': 'function',
    'function': {
        'name': 'multiply',
        'description': 'Multiply two numbers and return the product.',
        'parameters': { ... json schema, written by hand ... },
    },
}
```

Our server wrote none of that. Let's ask the client what the server derived from our type hints and
docstrings.

In [60]:
async def inspect_schemas() -> None:
    server = StdioServerParameters(command=sys.executable, args=['training_server.py'])

    async with Client(server) as client:
        listed = await client.list_tools()

        for tool in listed.tools:
            print('=' * 72)
            print(f'tool.name        : {tool.name}')
            print(f'tool.description : {tool.description}')
            print('tool.input_schema:')
            print(json.dumps(tool.input_schema, indent=2))
            print()


await inspect_schemas()

UnsupportedOperation: fileno

Compare `add_numbers` with the hand-written `multiply` schema from Lesson 4. Both tell a model the same
three things — *what the tool is called*, *what it does*, and *which arguments it takes* — but only one was
typed by a human.

> **Note:** depending on your SDK version, the printed JSON may use `inputSchema` instead of
> `input_schema`. Same content, older spelling.

### Look closely at one of them

One tool in our server has a description that is technically valid and practically useless:

```python
@mcp.tool()
def get_room_capacity(room_id: str) -> dict:
    '''Gets info.'''  ### FIXME: would a model pick this tool with a description like this?
```

Scroll up in the output and find it. The description `Gets info.` is what the **model** reads when it
decides which tool to call — it is not documentation for humans. In Lesson 4 we said *bad tool
descriptions lead to wrong tool choices*; MCP does not change that, it just publishes the bad description
to more places at once.

A description that does the job names the input and the outcome, and rules out the neighbours:

```python
def get_room_capacity(room_id: str) -> dict:
    '''Look up the seat capacity and projector availability of a training room by its id
    (such as lab_a or lab_b). Use this instead of guessing a room's size.'''
```

Fix it now, before the next step: the loop we are about to build depends on the model choosing the right
tool.

In [ ]:
# Fix the vague description and confirm the server publishes the better one.
from pathlib import Path

text = Path('training_server.py').read_text()
text = text.replace(
    "'''Gets info.'''",
    "'''Look up the seat capacity and projector availability of a training room by its id "
    "(such as lab_a or lab_b). Use this instead of guessing a room's size.'''",
)
Path('training_server.py').write_text(text)


async def show_room_tool_description() -> None:
    server = StdioServerParameters(command=sys.executable, args=['training_server.py'])
    async with Client(server) as client:
        listed = await client.list_tools()
        for tool in listed.tools:
            if tool.name == 'get_room_capacity':
                print('The model now reads:')
                print(' ', tool.description)


await show_room_tool_description()

# Step 4: MCP tools inside the tool-calling loop

This is where the two halves of the course meet.

In Lesson 4 the loop needed a hand-written schema and a hand-written registry:

```text
Lesson 4

  business_tools = [ {hand-written schema}, ... ]
  TOOL_REGISTRY  = { 'get_order_status': get_order_status, ... }
                          ^                     ^
                          |                     |
                    written by hand        the real function
```

Now both come from the server:

```text
Lesson 8

  tools  = await client.list_tools()      ->  schemas, discovered
  result = await client.call_tool(...)    ->  execution, over the protocol
```

So the loop itself barely changes. We only need one adapter: a function that converts an MCP tool
definition into the OpenAI tool format.

In [ ]:
def mcp_tool_to_openai_schema(tool) -> dict:
    '''Convert an MCP tool definition into the OpenAI tool format.'''
    schema = dict(tool.input_schema or {'type': 'object', 'properties': {}})
    schema.pop('title', None)

    return {
        'type': 'function',
        'function': {
            'name': tool.name,
            'description': tool.description,
            'parameters': schema,
        },
    }


print('Adapter ready: it is the bridge between the MCP shape and the OpenAI shape.')

### The loop, adapted

Same structure as Lesson 4 and Lesson 5: call the model, check for tool calls, execute, send the results
back, call the model again. Two things changed:

- Tools are **discovered** from the server instead of registered by hand.
- Execution goes **over the protocol** instead of calling a local function.

Everything else — including the error handling for unknown tools and malformed arguments — is the same
idea, and the `max_turns` guard is the one from Lesson 5.

In [63]:
import json
import os
from dotenv import load_dotenv
from fastmcp import Client
from utils.open_ai import OpenAI

# Import the server instance directly from training_server.py
from training_server import mcp as mcp_server

load_dotenv(override=False)

llm_client = OpenAI()
MODEL = os.environ.get('MODEL', 'gpt-4.1-mini')


async def ask_with_mcp_tools(question: str, max_turns: int = 5) -> str:
    # Connect directly to the imported server instance
    async with Client(mcp_server) as mcp_client:
        raw_tools = await mcp_client.list_tools()
        mcp_tools = raw_tools if isinstance(raw_tools, list) else raw_tools.tools

        # Format FastMCP tools for OpenAI tool calling
        openai_tools = []
        for tool in mcp_tools:
            parameters = getattr(tool, "inputSchema", None) or getattr(tool, "parameters", None) or {}
            openai_tools.append({
                "type": "function",
                "function": {
                    "name": tool.name,
                    "description": tool.description or "",
                    "parameters": parameters,
                }
            })

        print('Tools discovered from training_server.py:')
        for tool in mcp_tools:
            print(f'  - {tool.name}')
        print()

        messages = [
            {
                'role': 'system',
                'content': (
                    'You are a helpful training operations assistant. '
                    'Use the available tools for calculations and room lookups. '
                    'Do not do arithmetic mentally if a tool can do it.'
                ),
            },
            {'role': 'user', 'content': question},
        ]

        for turn in range(max_turns):
            response = llm_client.chat.completions.create(
                model=MODEL,
                messages=messages,
                tools=openai_tools,
                tool_choice='auto',
            )

            assistant_message = response.choices[0].message
            tool_calls = assistant_message.tool_calls or []

            if not tool_calls:
                return assistant_message.content or ""

            print(f'Turn {turn + 1}: requested {len(tool_calls)} tool call(s).')
            messages.append(assistant_message)

            for tool_call in tool_calls:
                tool_name = tool_call.function.name
                
                try:
                    arguments = json.loads(tool_call.function.arguments)
                except json.JSONDecodeError:
                    tool_result = {'error': 'Invalid JSON arguments.'}
                else:
                    print(f'  -> calling {tool_name} with {arguments} OVER MCP')
                    called = await mcp_client.call_tool(tool_name, arguments)

                    # Extract result and handle Pydantic model conversion
                    raw_data = getattr(called, "data", called)
                    if hasattr(raw_data, "model_dump"):
                        tool_result = raw_data.model_dump()
                    elif hasattr(raw_data, "dict"):
                        tool_result = raw_data.dict()
                    elif hasattr(called, "content") and called.content:
                        tool_result = called.content[0].text
                    else:
                        tool_result = raw_data

                print(f'     result: {tool_result}')

                # Safely serialize using default=str
                messages.append(
                    {
                        'role': 'tool',
                        'tool_call_id': tool_call.id,
                        'content': json.dumps(tool_result, default=str),
                    }
                )

        return 'The loop stopped before producing a final answer.'

### Run it

Three questions, chosen for what they exercise:

1. A calculation the model cannot do reliably on its own.
2. A room lookup, a tool it has never seen before this class.
3. A room that does not exist — the server's error path, crossing the protocol into the model.

In [64]:
questions = [
    'We have 25 trainees and 6 hours of training each. The cost is 40 per trainee-hour. What is the total training cost?',
    'How many people fit in lab_b?',
    'Does lab_z have a projector?',
]

for question in questions:
    print('=' * 80)
    print('User:', question)
    answer = await ask_with_mcp_tools(question)
    print('Assistant:', answer)
    print()

User: We have 25 trainees and 6 hours of training each. The cost is 40 per trainee-hour. What is the total training cost?
Tools discovered from training_server.py:
  - add_numbers
  - multiply_numbers
  - divide_numbers
  - get_room_capacity

Turn 1: requested 1 tool call(s).
  -> calling multiply_numbers with {'a': 25, 'b': 6} OVER MCP
     result: 150.0
Turn 2: requested 1 tool call(s).
  -> calling multiply_numbers with {'a': 150, 'b': 40} OVER MCP
     result: 6000.0
Assistant: The total training cost for 25 trainees, each receiving 6 hours of training at a rate of 40 per trainee-hour, is 6000.

User: How many people fit in lab_b?
Tools discovered from training_server.py:
  - add_numbers
  - multiply_numbers
  - divide_numbers
  - get_room_capacity

Turn 1: requested 1 tool call(s).
  -> calling get_room_capacity with {'room_id': 'lab_b'} OVER MCP
     result: {
  "found": true,
  "room_id": "lab_b",
  "capacity": 24,
  "has_projector": true
}
Assistant: Lab B can fit 24 people.

U

Watch the lines in the middle of each block: `calling ... OVER MCP`. The arithmetic never happened in this
notebook. The tools ran in a **separate process**, reached over stdin/stdout, and the model never knew.

That is the value of MCP in one sentence: *the model asks, the host decides, the server computes* — and
none of them had to share code.

One thing to notice in the last block: the server answered *"Room not found"* and the model handled it. We
did not write a retry, a fallback, or a special case for that — a structured error travelled the protocol
and the model read it, exactly like the error dictionaries in Lesson 4.

## Step 5: What MCP changes, and what it does not

### What it changes

| Concern | Before (Lessons 4 and 5) | With MCP |
| --- | --- | --- |
| Where tools live | Inside your program | In a separate server process |
| How they are described | Schema written by hand | Derived from type hints and docstrings |
| Who can use them | Only your program | Any MCP-compatible host |
| Discovery | You import and register | `client.list_tools()` |
| Execution | A Python function call | A protocol call, in another process |
| Sharing | Copy the code | Give the host a command or a URL |

### What it does not change

- **The model still does not execute anything.** It requests a tool call; the host decides whether to run
  it. Same contract as Lesson 4.
- **Tools are still code with permissions.** A tool that deletes files is exactly as dangerous on the other
  side of a protocol — possibly more so, because now other applications can reach it.
- **Tool results still need validation.** A failure can arrive as a normal-looking result with an error flag
  instead of an exception.
- **Good descriptions still matter more than good code.** The model picks tools by reading descriptions,
  and MCP publishes those descriptions even further.

### Security: what to keep in mind

Lesson 4 split tools into read-only and write. MCP adds a second question: **who is allowed to connect?**

```text
read-only tools, local server     ->  low risk, still least privilege
read-only tools, remote server    ->  think about auth and data exposure
write tools, any server           ->  confirmation steps, logging, least privilege
credentialed tools                ->  the child process does NOT inherit your env;
                                      secrets must be passed explicitly and kept out of logs
```

Also note that a server publishes **descriptions**, and those descriptions go into the model's prompt.
Publishing a server means publishing text into somebody else's context — a good reason to write tool
descriptions as carefully as you would write a system prompt.

## 6. Mini exercises

Try these exercises during class or as homework.

### Exercise 1

Add a tool:

```python
def count_free_seats(room_id: str, registered: int) -> dict
```

It should report how many seats remain in a room after a given number of registrations. Restart the client
and check that `list_tools()` shows it.

In [ ]:
# Your Exercise 1 code here.

### Exercise 2

Add a second resource, `training://rooms`, that returns every room with its capacity. Read it from the
client and print it.

In [ ]:
# Your Exercise 2 code here.

### Exercise 3

`divide_numbers` returns a structured error when `b` is zero. Rewrite `multiply_numbers` so it also returns
a structured error when either argument is missing or not a number, and verify the model can recover.

In [ ]:
# Your Exercise 3 code here.

### Exercise 4

Add a `training_plan_short` prompt that takes two arguments (`topic` and `minutes`) and renders a plan that
fits the requested duration.

In [ ]:
# Your Exercise 4 code here.

### Exercise 5

Make the client talk to **two** servers in the same run: `training_server.py` and a second server of your
own that exposes a single tool. Print the tools from both, and confirm the names never collide.

In [ ]:
# Your Exercise 5 code here.

### Exercise 6: Explain in your own words

Answer:

1. What is the difference between a **host**, a **client**, and a **server** in MCP?
2. Why does the resource belong to the application and the tool belong to the model?
3. What changes in the Lesson 4 loop when the tools come from MCP — and what does not?
4. Why does the model still not execute anything, even with a protocol in the middle?

## 7. Key Takeaways

- MCP standardizes how programs publish tools, resources, and prompts, so any compatible host can use them.
- The architecture is **host + client + server**, connected by a transport: `stdio` for local servers,
  streamable HTTP for deployed ones.
- A server's tool schemas are derived from **type hints and docstrings** — the same input the `@tool`
  decorator consumed in Lesson 5, now published over a protocol.
- The client's lifecycle is one block: `async with Client(transport) as client`.
- `list_tools()` gives you definitions; `call_tool()` gives you `content`, `structured_content`, and
  `is_error`. A failing tool is a **result**, not an exception.
- The Lesson 4 loop still works: discovering tools and executing them over MCP changes *where* the code
  runs, not *how* the loop is built.
- The model still never executes anything. MCP moves the boundary; it does not remove it.

## Optional Next Steps

After this class, good follow-up topics are:

- **Connect the server to a real host** (VS Code, Claude Desktop, Cursor). Every host takes the same launch
  command in its own config file — no code changes on your side:

  ```json
  {
    "mcpServers": {
      "training-ops": {
        "command": "/absolute/path/to/ai_env/bin/python",
        "args": ["/absolute/path/to/training_server.py"]
      }
    }
  }
  ```

  The path must be absolute: the host launches the server from its own working directory, not yours. In the
  AI Lab this may be blocked by machine policy — the value of the exercise is understanding the shape of
  the configuration, which is identical everywhere.
- **Remote servers over streamable HTTP**, for tools more than one person should reach.
- **Delegating the Lesson 7 agent's tools to an MCP server**, so the retriever lives outside the agent process.
- **Authentication and authorization** for remote servers: who may call which tool, and how that is proven.
- **The Inspector** (`mcp dev`) as a permanent debugging tool while developing servers.

--------------------------------------------------------------------------------------

*Instructor note — raw protocol capture (hide before class).*

If you want students to **see** the JSON-RPC on the wire, connect with a legacy client (`mode='legacy'`)
and log the first messages. This is what discovery and a call look like:

```json
--> {"jsonrpc":"2.0","id":1,"method":"initialize",
     "params":{"protocolVersion":"2025-11-25","capabilities":{},"clientInfo":{"name":"capture","version":"1.0"}}}
<-- {"jsonrpc":"2.0","id":1,"result":{"protocolVersion":"2025-11-25","capabilities":{"tools":{},"resources":{}},
     "serverInfo":{"name":"Training Ops","version":"..."}}}

--> {"jsonrpc":"2.0","method":"notifications/initialized"}
--> {"jsonrpc":"2.0","id":2,"method":"tools/list","params":{}}
<-- {"jsonrpc":"2.0","id":2,"result":{"tools":[{"name":"add_numbers","description":"Add two numbers and return the result.",
     "inputSchema":{"type":"object","properties":{"a":{"type":"number"},"b":{"type":"number"}},
     "required":["a","b"]}}]}}

--> {"jsonrpc":"2.0","id":3,"method":"tools/call",
     "params":{"name":"add_numbers","arguments":{"a":12,"b":8}}}
<-- {"jsonrpc":"2.0","id":3,"result":{"content":[{"type":"text","text":"20"}],
     "structuredContent":20,"isError":false}}
```

Three things worth pointing out on the slide:

1. It is **JSON-RPC 2.0**: a `method`, `params`, and a matched `id`. Nothing exotic.
2. The `inputSchema` in `tools/list` is the same shape we hand to the model in Lesson 4 — just moved to
   where every host can read it.
3. A failing tool comes back as a **result** with `isError: true`, never as a transport error.

This block is instructor material: do not leave the raw capture in the student-facing notebook.